In [7]:
import datetime

# Exemplo de estrutura de dados para o teu código
regre_negocio = """
REGRAS DO ESTABELECIMENTO:
1. IDADE: Aceitamos apenas crianças até aos 7 anos de idade.
2. HORÁRIO REGULAR: O serviço corre entre as 15:00/16:00 (3h-4h da tarde) até às 17:00.
3. FINS DE SEMANA: Existe a possibilidade de horas extras, mas apenas sob consulta prévia.
4. SEGURANÇA (OBRIGATÓRIO): Não processar nenhuma reserva sem que o utilizador forneça:
   - Contactos de emergência.
   - Lista de alergias detalhada.
   - Restrições alimentares (o que pode ou não comer).
   - Idade confirmada da criança.
"""

def validar_reserva(idade, horario_inicio, horario_fim, is_weekend, tem_contactos, tem_alergias, tem_restricoes_alimentares):
    erros = []
    itens_em_falta = []
    dados_entregues = []

    # 1. Validação de Idade
    if idade is None:
        itens_em_falta.append("Idade da criança.")
    elif idade > 7:
        erros.append("Erro: Só aceitamos crianças até aos 7 anos.")
    else:
        dados_entregues.append(f"Idade da criança: {idade} anos.")

    # 2. Validação de Horário
    try:
        inicio_dt = datetime.datetime.strptime(horario_inicio, "%H:%M").time()
        fim_dt = datetime.datetime.strptime(horario_fim, "%H:%M").time()
        hora_limite_inicio_regular = datetime.time(15, 0)
        hora_limite_inicio_alternativa = datetime.time(16, 0)
        hora_limite_fim_regular = datetime.time(17, 0)

        if not is_weekend:
            # Para dias de semana, o início deve ser entre 15h e 16h, e o fim às 17h.
            if not (inicio_dt >= hora_limite_inicio_regular and inicio_dt <= hora_limite_inicio_alternativa and fim_dt == hora_limite_fim_regular):
                erros.append(f"Erro: Para dias de semana, o horário regular é das 15h/16h às 17h. O horário solicitado ({horario_inicio}-{horario_fim}) não é válido.")
        # Se for fim de semana, qualquer horário é potencialmente válido, mas as "horas extras" devem ser consultadas.
        # A validação estrita de horário não se aplica da mesma forma aqui, apenas para dias de semana.

        dados_entregues.append(f"Horário da reserva: {horario_inicio} às {horario_fim} (Fim de semana: {'Sim' if is_weekend else 'Não'}).")

    except ValueError:
        erros.append("Erro: Formato de horário inválido. Utilize HH:MM para início e fim.")
    except TypeError: # se horario_inicio ou horario_fim for None ou outro tipo inesperado
        erros.append("Erro: Horário de início e fim são obrigatórios.")


    # 3. Check-list de Dados Obrigatórios
    if tem_contactos:
        dados_entregues.append("Contactos de emergência.")
    else:
        itens_em_falta.append("Contactos de emergência.")

    if tem_alergias:
        dados_entregues.append("Lista de alergias detalhada.")
    else:
        itens_em_falta.append("Lista de alergias detalhada.")

    if tem_restricoes_alimentares:
        dados_entregues.append("Restrições alimentares.")
    else:
        itens_em_falta.append("Restrições alimentares.")


    # Resumo Final
    response = ""
    if erros:
        response += "\n".join(erros)
    else:
        if itens_em_falta:
            response += "Não foi possível pré-validar a reserva. Itens em falta:\n- " + "\n- ".join(itens_em_falta)
        else:
            response += "Reserva pré-validada com sucesso!\n"
            response += "Dados recebidos:\n- " + "\n- ".join(dados_entregues)
            if is_weekend:
                response += "\nNota: Por ser fim de semana e o horário poder ser fora do regular, as horas extras estão sujeitas a confirmação prévia."

    return response

# --- Testes Rápidos --- #
print("--- Teste 1: Reserva Válida (Dia de Semana) ---")
print(validar_reserva(idade=5, horario_inicio="15:30", horario_fim="17:00", is_weekend=False,
                      tem_contactos=True, tem_alergias=True, tem_restricoes_alimentares=True))
print("\n")

print("--- Teste 2: Idade acima do limite ---")
print(validar_reserva(idade=8, horario_inicio="15:30", horario_fim="17:00", is_weekend=False,
                      tem_contactos=True, tem_alergias=True, tem_restricoes_alimentares=True))
print("\n")

print("--- Teste 3: Horário Inválido (Dia de Semana) ---")
print(validar_reserva(idade=5, horario_inicio="14:00", horario_fim="16:00", is_weekend=False,
                      tem_contactos=True, tem_alergias=True, tem_restricoes_alimentares=True))
print("\n")

print("--- Teste 4: Faltam Dados Obrigatórios ---")
print(validar_reserva(idade=6, horario_inicio="15:00", horario_fim="17:00", is_weekend=False,
                      tem_contactos=False, tem_alergias=True, tem_restricoes_alimentares=False))
print("\n")

print("--- Teste 5: Reserva de Fim de Semana (Sujeita a Confirmação) ---")
print(validar_reserva(idade=4, horario_inicio="10:00", horario_fim="12:00", is_weekend=True,
                      tem_contactos=True, tem_alergias=False, tem_restricoes_alimentares=True))
print("\n")

print("--- Teste 6: Idade não fornecida ---")
print(validar_reserva(idade=None, horario_inicio="15:00", horario_fim="17:00", is_weekend=False,
                      tem_contactos=True, tem_alergias=True, tem_restricoes_alimentares=True))
print("\n")

print("--- Teste 7: Horário formato inválido ---")
print(validar_reserva(idade=5, horario_inicio="15:0", horario_fim="17:00", is_weekend=False,
                      tem_contactos=True, tem_alergias=True, tem_restricoes_alimentares=True))
print("\n")

--- Teste 1: Reserva Válida (Dia de Semana) ---
Reserva pré-validada com sucesso!
Dados recebidos:
- Idade da criança: 5 anos.
- Horário da reserva: 15:30 às 17:00 (Fim de semana: Não).
- Contactos de emergência.
- Lista de alergias detalhada.
- Restrições alimentares.


--- Teste 2: Idade acima do limite ---
Erro: Só aceitamos crianças até aos 7 anos.


--- Teste 3: Horário Inválido (Dia de Semana) ---
Erro: Para dias de semana, o horário regular é das 15h/16h às 17h. O horário solicitado (14:00-16:00) não é válido.


--- Teste 4: Faltam Dados Obrigatórios ---
Não foi possível pré-validar a reserva. Itens em falta:
- Contactos de emergência.
- Restrições alimentares.


--- Teste 5: Reserva de Fim de Semana (Sujeita a Confirmação) ---
Não foi possível pré-validar a reserva. Itens em falta:
- Lista de alergias detalhada.


--- Teste 6: Idade não fornecida ---
Não foi possível pré-validar a reserva. Itens em falta:
- Idade da criança.


--- Teste 7: Horário formato inválido ---
Reserva p

In [1]:
import datetime

# --- Início da Interação de Reserva ---

print("Olá! Bem-vindo(a) ao serviço 'Kids & Pets Care'.")

# Initialize variables
is_child = None
nome = ""
idade = None
animal_species = None
animal_breed = None
animal_age = None
horario_inicio = ""
horario_fim = ""
is_weekend = False
tem_contactos = False
tem_alergias = False
tem_restricoes_alimentares = False


# Passo 1: Pergunta se o serviço é para uma CRIANÇA ou para um ANIMAL.
while True:
    tipo_servico_str = input("É para uma **CRIANÇA** (👶) ou para um **ANIMAL** (🐾)? Digite 'C' ou 'A': ").lower()
    if tipo_servico_str == 'c':
        is_child = True
        nome = input("Por favor, diga-me o nome da criança: ")
        while True:
            try:
                idade_str = input(f"Qual a idade de {nome} (👶)? ")
                idade = int(idade_str)
                if idade > 7:
                    print(f"Lamentamos, mas o nosso serviço de babysitting é exclusivo para crianças até aos **7 anos de idade**. A reserva para **{nome}** não pode ser processada. Tenha um bom dia.")
                    exit()
                break # Exit age loop if valid
            except ValueError:
                print("Erro: Idade inválida. A reserva não pode ser processada. Tenha um bom dia.")
                exit()
        break # Exit service type loop
    elif tipo_servico_str == 'a':
        is_child = False
        nome = input("Por favor, diga-me o nome do animal: ")
        animal_species = input(f"Qual a espécie de {nome} (🐾)? (Ex: Cão, Gato, Pássaro) ")
        animal_breed = input(f"Qual a raça de {nome} (🐾)? ")
        while True:
            try:
                animal_age_str = input(f"Qual a idade de {nome} (🐾)? ")
                animal_age = int(animal_age_str)
                break # Exit age loop if valid
            except ValueError:
                print("Erro: Idade inválida. A reserva não pode ser processada. Tenha um bom dia.")
                exit()
        break # Exit service type loop
    else:
        print("Erro: Tipo de serviço inválido. A reserva não pode ser processada. Tenha um bom dia.")
        exit()

# Passo 2: Pergunta o dia da semana e o horário
print("\nÓtimo! Agora, vamos falar sobre o horário.")
while True:
    is_weekend_str = input("A reserva é para um **fim de semana** (s/n)? ").lower()
    if is_weekend_str in ['s', 'n']:
        is_weekend = True if is_weekend_str == 's' else False
        break
    else:
        print("Erro: Resposta inválida para fim de semana. Por favor, responda 's' para sim ou 'n' para não. A reserva não pode ser processada.")
        exit()

while True:
    horario_inicio = input("Qual a hora de **início** pretendida (formato HH:MM)? ")
    horario_fim = input("Qual a hora de **fim** pretendida (formato HH:MM)? ")
    try:
        datetime.datetime.strptime(horario_inicio, "%H:%M")
        datetime.datetime.strptime(horario_fim, "%H:%M")
        break
    except ValueError:
        print("Erro: Formato de horário inválido. Utilize HH:MM para início e fim. A reserva não pode ser processada.")
        exit()

# Passo 3: Solicita os Contactos de Emergência
print("\nPara a segurança e bem-estar, precisamos de algumas informações essenciais.")
while True:
    tem_contactos_str = input("Já forneceu os **contactos de emergência** (s/n)? ").lower()
    if tem_contactos_str in ['s', 'n']:
        tem_contactos = True if tem_contactos_str == 's' else False
        if not tem_contactos:
            print("⚠️ **Os contactos de emergência são obrigatórios para prosseguir com a reserva.** A reserva não pode ser processada. Tenha um bom dia.")
            exit()
        else:
            break
    else:
        print("Erro: Resposta inválida para contactos de emergência. Por favor, responda 's' para sim ou 'n' para não. A reserva não pode ser processada.")
        exit()

# Passo 4: Pergunta sobre ALERGIAS e ALIMENTAÇÃO
while True:
    tem_alergias_str = input(f"Tem uma lista detalhada de **alergias** {'da criança' if is_child else 'do animal'} (s/n)? ").lower()
    if tem_alergias_str in ['s', 'n']:
        tem_alergias = True if tem_alergias_str == 's' else False
        if not tem_alergias:
            print(f"⚠️ **As informações sobre alergias são obrigatórias para a segurança e bem-estar {'da criança' if is_child else 'do animal'}.** A reserva não pode ser processada. Tenha um bom dia.")
            exit()
        else:
            break
    else:
        print("Erro: Resposta inválida para alergias. Por favor, responda 's' para sim ou 'n' para não. A reserva não pode ser processada.")
        exit()

while True:
    tem_restricoes_str = input(f"Tem informações sobre **restrições alimentares** (o que pode ou não comer) {'da criança' if is_child else 'do animal'} (s/n)? ").lower()
    if tem_restricoes_str in ['s', 'n']:
        tem_restricoes_alimentares = True if tem_restricoes_str == 's' else False
        if not tem_restricoes_alimentares:
            print(f"⚠️ **As informações sobre restrições alimentares são obrigatórias para a segurança e bem-estar {'da criança' if is_child else 'do animal'}.** A reserva não pode ser processada. Tenha um bom dia.")
            exit()
        else:
            break
    else:
        print("Erro: Resposta inválida para restrições alimentares. Por favor, responda 's' para sim ou 'n' para não. A reserva não pode ser processada.")
        exit()


# Passo 5: Gera um "CARTÃO DE RESERVA" final com todos os dados para confirmação do cliente.
print("\n--- 📝 **CARTÃO DE RESERVA** ---\n")
print(f"**Serviço:** {'👶 Babysitting' if is_child else '🐾 Petsitting'}")
print(f"**Nome:** {nome}")
if is_child:
    print(f"**Idade:** {idade} anos")
else:
    print(f"**Espécie:** {animal_species}")
    print(f"**Raça:** {animal_breed}")
    print(f"**Idade:** {animal_age} anos")
print(f"**Horário:** {horario_inicio} - {horario_fim} (Fim de semana: {'Sim' if is_weekend else 'Não'}) ")
print(f"**Contactos de emergência fornecidos:** {'Sim ✅' if tem_contactos else 'Não ❌'}")
print(f"**Alergias detalhadas fornecidas:** {'Sim ✅' if tem_alergias else 'Não ❌'}")
print(f"**Restrições alimentares fornecidas:** {'Sim ✅' if tem_restricoes_alimentares else 'Não ❌'}")

print("\nA processar a sua pré-reserva...")

# Validação condicional baseada no tipo de serviço
if is_child:
    resultado_validacao = validar_reserva(
        idade=idade,
        horario_inicio=horario_inicio,
        horario_fim=horario_fim,
        is_weekend=is_weekend,
        tem_contactos=tem_contactos,
        tem_alergias=tem_alergias,
        tem_restricoes_alimentares=tem_restricoes_alimentares
    )
    print("\n" + resultado_validacao)
else:
    # Para petsitting, a função `validar_reserva` é específica para crianças.
    # Simulamos uma validação baseada na recolha dos dados obrigatórios.
    if tem_contactos and tem_alergias and tem_restricoes_alimentares:
        response_pet = "Reserva pré-validada com sucesso para Petsitting! Entraremos em contacto para confirmar os detalhes.\n"
        response_pet += "Dados recebidos:\n"
        response_pet += f"- Nome: {nome}\n- Espécie: {animal_species}\n- Raça: {animal_breed}\n- Idade: {animal_age} anos\n"
        response_pet += f"- Horário: {horario_inicio} às {horario_fim} (Fim de semana: {'Sim' if is_weekend else 'Não'}).\n"
        response_pet += "- Contactos de emergência.\n- Lista de alergias detalhada.\n- Restrições alimentares."
        if is_weekend:
            response_pet += "\nNota: Por ser fim de semana, o horário está sujeito a confirmação prévia de disponibilidade."
        print(response_pet)
    else:
        print("⚠️ **Não foi possível pré-validar a reserva de Petsitting. Há informações obrigatórias em falta.**")
        missing_pet_items = []
        if not tem_contactos: missing_pet_items.append("Contactos de emergência")
        if not tem_alergias: missing_pet_items.append("Lista de alergias/doenças")
        if not tem_restricoes_alimentares: missing_pet_items.append("Restrições alimentares")
        print("- " + "\n- ".join(missing_pet_items))


print("\n--- Fim da Interação ---")

Olá! Bem-vindo(a) ao serviço 'Kids & Pets Care'.
É para uma **CRIANÇA** (👶) ou para um **ANIMAL** (🐾)? Digite 'C' ou 'A': c
Por favor, diga-me o nome da criança: Lika Damas
Qual a idade de Lika Damas (👶)? 9
Lamentamos, mas o nosso serviço de babysitting é exclusivo para crianças até aos **7 anos de idade**. A reserva para **Lika Damas** não pode ser processada. Tenha um bom dia.

Ótimo! Agora, vamos falar sobre o horário.


KeyboardInterrupt: Interrupted by user

In [1]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# --- ESTILIZAÇÃO (CSS para aspeto de Site Profissional) ---
display(HTML("""
<style>
    .main-container { padding: 20px; background-color: #f8f9fa; border-radius: 10px; border: 1px solid #e0e0e0; }
    .header { color: #2c3e50; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; text-align: center; }
    .footer { font-size: 12px; color: #7f8c8d; text-align: center; margin-top: 20px; }
    .error-msg { color: #e74c3c; font-weight: bold; padding: 10px; background: #fdeaea; border-radius: 5px; }
    .success-msg { color: #27ae60; font-weight: bold; padding: 20px; background: #eafaf1; border-radius: 5px; border-left: 5px solid #27ae60; }
</style>
"""))

# --- COMPONENTES DA INTERFACE ---
header = widgets.HTML("<h1 class='header'>✨ Kids & Pets Care</h1><p style='text-align:center;'>Reserva de Babysitting e Petsitting</p><hr>")

tipo_servico = widgets.Dropdown(options=[('👶 Babysitting', 'baby'), ('🐾 Petsitting', 'pet')], description='Serviço:', style={'description_width': 'initial'})
nome_cli = widgets.Text(placeholder='O seu nome completo', description='Responsável:', style={'description_width': 'initial'})
idade = widgets.IntText(value=0, description='Idade (Anos):', style={'description_width': 'initial'})
dia = widgets.Dropdown(options=[('Segunda a Sexta (Horário Regular)', 'util'), ('Fim de Semana (Extra sob consulta)', 'fds')], description='Dia:', style={'description_width': 'initial'})
horario = widgets.SelectionSlider(options=['15:00', '16:00', '17:00'], description='Início:', style={'description_width': 'initial'})
contacto = widgets.Text(placeholder='Telemóvel de Emergência', description='Emergência:', style={'description_width': 'initial'})
alergias = widgets.Textarea(placeholder='Liste todas as alergias ou escreva "Nenhuma"', description='Alergias:', style={'description_width': 'initial'})
dieta = widgets.Textarea(placeholder='O que pode ou não comer?', description='Dieta/Restrições:', style={'description_width': 'initial'})

btn_reservar = widgets.Button(description="Confirmar Reserva", button_style='primary', layout=widgets.Layout(width='100%', height='40px'))
output = widgets.Output()

# --- LÓGICA DE VALIDAÇÃO ---
def validar_e_enviar(b):
    with output:
        clear_output()

        # 1. Validação de Idade para Crianças
        if tipo_servico.value == 'baby' and idade.value > 7:
            display(HTML("<p class='error-msg'>⚠️ Atenção: Por motivos de segurança e especialização, apenas aceitamos crianças até aos 7 anos.</p>"))
            return

        # 2. Verificação de Campos Obrigatórios
        campos_vazio = not nome_cli.value or not contacto.value or not alergias.value or not dieta.value
        if campos_vazio:
            display(HTML("<p class='error-msg'>⚠️ Erro: Todos os campos (Responsável, Emergência, Alergias e Dieta) são obrigatórios.</p>"))
            return

        # 3. Sucesso - Resumo Final
        servico_nome = "Babysitting" if tipo_servico.value == 'baby' else "Petsitting"
        display(HTML(f"""
            <div class='success-msg'>
                <h3>✅ Pedido de Reserva Enviado!</h3>
                <p><b>Serviço:</b> {servico_nome}</p>
                <p><b>Responsável:</b> {nome_cli.value}</p>
                <p><b>Idade:</b> {idade.value} anos</p>
                <p><b>Horário:</b> {horario.value} às 17:00 ({dia.label})</p>
                <hr>
                <p><b>Protocolo de Segurança:</b></p>
                <ul>
                    <li>Emergência: {contacto.value}</li>
                    <li>Alergias: {alergias.value}</li>
                    <li>Dieta: {dieta.value}</li>
                </ul>
                <p><small>Entraremos em contacto em breve para confirmar a disponibilidade final.</small></p>
            </div>
        """))

btn_reservar.on_click(validar_e_enviar)

# --- LAYOUT FINAL ---
formulario = widgets.VBox([
    header,
    widgets.HBox([tipo_servico, idade]),
    nome_cli,
    widgets.HBox([dia, horario]),
    contacto,
    alergias,
    dieta,
    btn_reservar,
    widgets.HTML("<p class='footer'>🔒 Os seus dados estão seguros e serão usados apenas para fins de emergência.</p>")
], layout=widgets.Layout(max_width='600px', margin='0 auto', padding='20px'))

display(formulario, output)

Output()

In [2]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# --- ESTILIZAÇÃO PROFISSIONAL ---
display(HTML("""
<style>
    .main-box { border: 2px solid #e0e0e0; border-radius: 15px; padding: 25px; background-color: #ffffff; font-family: 'Helvetica', sans-serif; }
    .header-title { color: #2c3e50; text-align: center; margin-bottom: 5px; }
    .header-subtitle { color: #7f8c8d; text-align: center; font-size: 14px; margin-bottom: 20px; }
    .error-box { background-color: #ffeeee; border-left: 5px solid #ff4444; padding: 10px; color: #cc0000; border-radius: 4px; margin-top: 10px; }
    .success-box { background-color: #eafff0; border-left: 5px solid #2ecc71; padding: 20px; color: #27ae60; border-radius: 4px; margin-top: 10px; }
    .section-title { font-weight: bold; color: #34495e; margin-top: 10px; border-bottom: 1px solid #eee; padding-bottom: 5px; }
</style>
"""))

# --- COMPONENTES GERAIS (CLIENTE) ---
nome_responsavel = widgets.Text(placeholder="Nome completo do tutor/pai", description="Responsável:", style={'description_width': 'initial'})
contacto_emergencia = widgets.Text(placeholder="Nº de Telemóvel urgente", description="Emergência:", style={'description_width': 'initial'})
dia_semana = widgets.Dropdown(options=[('Segunda a Sexta (15h-17h)', 'util'), ('Fim de Semana (Extra)', 'fds')], description="Período:", style={'description_width': 'initial'})
horario_inicio = widgets.SelectionSlider(options=['15:00', '16:00', '17:00'], description="Início:", style={'description_width': 'initial'})

# --- COMPONENTES ESPECÍFICOS: BABYSITTING ---
idade_crianca = widgets.IntText(value=0, description="Idade da Criança:", style={'description_width': 'initial'})
alergias_crianca = widgets.Textarea(placeholder="Alergias e restrições alimentares específicas", description="Saúde/Dieta:", style={'description_width': 'initial'})

# --- COMPONENTES ESPECÍFICOS: PETSITTING ---
especie_pet = widgets.Dropdown(options=['Cão', 'Gato', 'Ave', 'Roedor', 'Outro'], description="Espécie:", style={'description_width': 'initial'})
raca_pet = widgets.Text(placeholder="Ex: Labrador, Siamês...", description="Raça:", style={'description_width': 'initial'})
idade_pet = widgets.IntText(value=0, description="Idade do Pet:", style={'description_width': 'initial'})
cuidados_pet = widgets.Textarea(placeholder="Horários de comida, medicação ou comportamento (agressivo/medroso)", description="Cuidados/Dieta:", style={'description_width': 'initial'})

# --- LÓGICA DE INTERFACE ---
output = widgets.Output()
btn_confirmar = widgets.Button(description="FINALIZAR RESERVA", button_style='success', layout=widgets.Layout(width='100%', height='45px'))

# Organização por Abas
tab_baby = widgets.VBox([
    widgets.HTML("<div class='section-title'>Dados da Criança</div>"),
    idade_crianca,
    alergias_crianca
])

tab_pet = widgets.VBox([
    widgets.HTML("<div class='section-title'>Dados do Animal</div>"),
    widgets.HBox([especie_pet, raca_pet]),
    idade_pet,
    cuidados_pet
])

tabs = widgets.Tab(children=[tab_baby, tab_pet])
tabs.set_title(0, '👶 Babysitting')
tabs.set_title(1, '🐾 Petsitting')

# --- VALIDAÇÃO FINAL ---
def processar_formulario(b):
    with output:
        clear_output()

        # Validar dados comuns
        if not nome_responsavel.value or not contacto_emergencia.value:
            display(HTML("<div class='error-box'>⚠️ Por favor, preencha o nome do responsável e o contacto de emergência.</div>"))
            return

        # Lógica Babysitting
        if tabs.selected_index == 0:
            if idade_crianca.value > 7:
                display(HTML("<div class='error-box'>❌ Limite de Idade Excedido: Só aceitamos crianças até aos 7 anos.</div>"))
                return
            if not alergias_crianca.value:
                display(HTML("<div class='error-box'>⚠️ É obrigatório declarar as alergias e dieta da criança.</div>"))
                return

            # Sucesso Baby
            resumo = f"<b>Criança de {idade_crianca.value} anos</b><br>Alergias/Dieta: {alergias_crianca.value}"
            tipo = "Babysitting"

        # Lógica Petsitting
        else:
            if not raca_pet.value or not cuidados_pet.value:
                display(HTML("<div class='error-box'>⚠️ Por favor, indique a raça e os cuidados/dieta do animal.</div>"))
                return

            resumo = f"<b>{especie_pet.value} ({raca_pet.value})</b> - {idade_pet.value} anos<br>Cuidados: {cuidados_pet.value}"
            tipo = "Petsitting"

        # Mensagem Final de Confiança
        display(HTML(f"""
            <div class='success-box'>
                <h3>⭐ Reserva Solicitada com Sucesso!</h3>
                <p><b>Serviço:</b> {tipo}</p>
                <p><b>Responsável:</b> {nome_responsavel.value} (Emergência: {contacto_emergencia.value})</p>
                <p><b>Horário:</b> {horario_inicio.value} às 17:00 ({dia_semana.label})</p>
                <hr>
                <p>{resumo}</p>
                <p style='font-size: 12px;'><i>Enviamos um alerta para a nossa equipa. Entraremos em contacto em 15 minutos.</i></p>
            </div>
        """))

btn_confirmar.on_click(processar_formulario)

# --- EXIBIÇÃO ---
layout_final = widgets.VBox([
    widgets.HTML("<h1 class='header-title'>Kids & Pets Care</h1><p class='header-subtitle'>Segurança e Carinho para quem você ama</p>"),
    nome_responsavel,
    contacto_emergencia,
    widgets.HBox([dia_semana, horario_inicio]),
    tabs,
    widgets.HTML("<br>"),
    btn_confirmar,
    output
], layout=widgets.Layout(max_width='650px', margin='0 auto', border='1px solid #ddd', padding='20px', border_radius='10px'))

display(layout_final)

In [4]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# --- ESTILO DE SITE PREMIUM ---
display(HTML("""
<style>
    .main-card { border: 1px solid #dcdde1; border-radius: 15px; padding: 30px; background-color: #ffffff; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }
    .title { color: #2f3640; font-family: 'Helvetica Neue', sans-serif; text-align: center; font-weight: 700; }
    .input-label { font-weight: 600; color: #353b48; margin-bottom: 5px; }
    .error-alert { background-color: #fab1a0; color: #c0392b; padding: 15px; border-radius: 8px; margin-top: 15px; font-weight: bold; }
    .success-alert { background-color: #55efc4; color: #006266; padding: 20px; border-radius: 8px; margin-top: 15px; border-left: 6px solid #00b894; }
</style>
"""))

# --- CAMPOS GERAIS ---
tipo_servico = widgets.ToggleButtons(
    options=[('👶 Babysitting', 'baby'), ('🐾 Petsitting', 'pet')],
    description='O que precisa?',
    button_style='info',
    tooltips=['Crianças até 7 anos', 'Animais de estimação'],
)

nome_resp = widgets.Text(placeholder="Seu nome", description="Responsável:")
contacto = widgets.Text(placeholder="Telemóvel de emergência", description="Contacto:")
dia_tipo = widgets.Dropdown(options=[('Segunda a Sexta (15h-17h)', 'util'), ('Fim de Semana (Extra)', 'fds')], description="Dia:")
hora_inicio = widgets.Dropdown(options=['15:00', '16:00'], description="Hora Início:")

# --- CONTENTORES DINÂMICOS ---
container_especifico = widgets.VBox()
output = widgets.Output()

def atualizar_interface(change):
    with output:
        clear_output()

    if change['new'] == 'baby':
        # Campos exclusivos para Crianças
        idade_c = widgets.IntSlider(value=1, min=0, max=15, description="Idade:")
        saude_c = widgets.Textarea(placeholder="Alergias e o que PODE ou NÃO comer", description="Saúde/Dieta:")
        container_especifico.children = [
            widgets.HTML("<b style='color:#0984e3'>Configurações de Babysitting:</b>"),
            idade_c, saude_c
        ]
    else:
        # Campos exclusivos para Pets
        especie = widgets.Text(placeholder="Cão, Gato, etc.", description="Espécie:")
        raca = widgets.Text(placeholder="Raça do animal", description="Raça:")
        idade_p = widgets.IntText(value=1, description="Idade Pet:")
        dieta_p = widgets.Textarea(placeholder="Alergias e restrições alimentares do pet", description="Dieta/Saúde:")
        container_especifico.children = [
            widgets.HTML("<b style='color:#e67e22'>Configurações de Petsitting:</b>"),
            especie, raca, idade_p, dieta_p
        ]

tipo_servico.observe(atualizar_interface, names='value')
atualizar_interface({'new': 'baby'}) # Iniciar com Babysitting por defeito

# --- BOTÃO E VALIDAÇÃO ---
btn_finalizar = widgets.Button(description="SOLICITAR AGENDAMENTO", button_style='success', layout=widgets.Layout(width='100%', height='50px'))

def validar_final(b):
    with output:
        clear_output()

        # Validação comum
        if not nome_resp.value or not contacto.value:
            display(HTML("<div class='error-alert'>⚠️ Preencha o nome do responsável e o contacto!</div>"))
            return

        # Lógica Babysitting
        if tipo_servico.value == 'baby':
            idade = container_especifico.children[1].value
            saude = container_especifico.children[2].value
            if idade > 7:
                display(HTML("<div class='error-alert'>❌ Erro: Só aceitamos crianças até aos 7 anos.</div>"))
                return
            if not saude:
                display(HTML("<div class='error-alert'>⚠️ É obrigatório indicar as alergias e dieta.</div>"))
                return
            resumo = f"👶 <b>Criança:</b> {idade} anos<br><b>Saúde:</b> {saude}"

        # Lógica Petsitting
        else:
            esp = container_especifico.children[1].value
            raca = container_especifico.children[2].value
            id_p = container_especifico.children[3].value
            diet = container_especifico.children[4].value
            if not esp or not raca or not diet:
                display(HTML("<div class='error-alert'>⚠️ Preencha todos os dados do animal (Espécie, Raça e Dieta).</div>"))
                return
            resumo = f"🐾 <b>Pet:</b> {esp} ({raca}) - {id_p} anos<br><b>Dieta:</b> {diet}"

        # SUCESSO
        display(HTML(f"""
            <div class='success-alert'>
                <h3>✅ Solicitação Recebida com Sucesso!</h3>
                <p><b>Responsável:</b> {nome_resp.value} ({contacto.value})</p>
                <p><b>Horário:</b> {hora_inicio.value} às 17:00 ({dia_tipo.label})</p>
                <hr>
                <p>{resumo}</p>
                <p><i>A nossa equipa entrará em contacto para confirmar os detalhes finais.</i></p>
            </div>
        """))

btn_finalizar.on_click(validar_final)

# --- RENDERIZAÇÃO ---
layout_site = widgets.VBox([
    widgets.HTML("<h1 class='title'>Kids & Pets Care</h1><hr>"),
    tipo_servico,
    widgets.HTML("<br>"),
    nome_resp, contacto,
    widgets.HBox([dia_tipo, hora_inicio]),
    widgets.HTML("<br>"),
    container_especifico,
    widgets.HTML("<br>"),
    btn_finalizar,
    output
], layout=widgets.Layout(max_width='600px', margin='0 auto', padding='20px'))

display(layout_site)

In [6]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# --- ESTILO VISUAL DE APP PROFISSIONAL ---
display(HTML("""
<style>
    .main-box { border: 1px solid #d1d8e0; border-radius: 20px; padding: 30px; background-color: #ffffff; box-shadow: 0 8px 16px rgba(0,0,0,0.05); }
    .title { color: #4b6584; text-align: center; font-family: 'Segoe UI', sans-serif; font-weight: bold; }
    .section-header { color: #a5b1c2; font-size: 13px; text-transform: uppercase; letter-spacing: 1px; margin-top: 15px; border-bottom: 1px solid #f1f2f6; }
    .error-msg { background-color: #fed33022; color: #eb3b5a; padding: 15px; border-radius: 10px; border-left: 5px solid #eb3b5a; margin-top: 15px; }
    .success-card { background-color: #20bf6b11; color: #20bf6b; padding: 25px; border-radius: 15px; border: 1px solid #20bf6b33; margin-top: 20px; }
</style>
"""))

# --- CAMPOS DE SELEÇÃO INICIAL ---
tipo_servico = widgets.ToggleButtons(
    options=[('👶 Babysitting', 'baby'), ('🐾 Petsitting', 'pet')],
    description='Serviço:', button_style='primary'
)

# --- CAMPOS COMUNS ---
nome_resp = widgets.Text(placeholder="Nome do Pai/Mãe/Tutor", description="Responsável:")
contacto = widgets.Text(placeholder="Telemóvel de Emergência", description="Contacto:")
dia_semana = widgets.Dropdown(options=[('Segunda a Sexta (15h-17h)', 'util'), ('Fim de Semana (Extra)', 'fds')], description="Dia:")

# --- CONTENTOR PARA CAMPOS DINÂMICOS ---
container_especifico = widgets.VBox()
output = widgets.Output()

def atualizar_campos(change):
    with output: clear_output()

    if change['new'] == 'baby':
        # Campos exclusivos para Crianças
        nome_crianca = widgets.Text(placeholder="Nome da criança", description="Nome:")
        genero = widgets.Dropdown(options=[('Menino', 'm'), ('Menina', 'f')], description="Género:")
        idade_c = widgets.IntText(value=0, description="Idade:")
        gostos = widgets.Textarea(placeholder="O que gosta de fazer? (Desenhar, brincar lá fora, ler...)", description="Interesses:")
        saude_c = widgets.Textarea(placeholder="ALERGIAS e o que pode/não pode comer (OBRIGATÓRIO)", description="Saúde/Dieta:")

        container_especifico.children = [
            widgets.HTML("<div class='section-header'>Informações da Criança</div>"),
            nome_crianca, genero, idade_c, gostos, saude_c
        ]
    else:
        # Campos exclusivos para Pets
        especie = widgets.Text(placeholder="Cão, Gato, etc.", description="Espécie:")
        raca = widgets.Text(placeholder="Raça", description="Raça:")
        idade_p = widgets.IntText(value=0, description="Idade Pet:")
        cuidados = widgets.Textarea(placeholder="O que o animal gosta de fazer e restrições de dieta/saúde", description="Detalhes:")

        container_especifico.children = [
            widgets.HTML("<div class='section-header'>Informações do Animal</div>"),
            especie, raca, idade_p, cuidados
        ]

tipo_servico.observe(atualizar_campos, names='value')
atualizar_campos({'new': 'baby'}) # Começa com babysitting

# --- BOTÃO E VALIDAÇÃO ---
btn = widgets.Button(description="SUBMETER RESERVA", button_style='success', layout=widgets.Layout(width='100%', height='50px'))

def validar_e_exibir(b):
    with output:
        clear_output()

        # 1. Validar Responsável
        if not nome_resp.value or not contacto.value:
            display(HTML("<div class='error-msg'>⚠️ Falta o nome do responsável e contacto de emergência.</div>"))
            return

        # 2. Lógica Babysitting
        if tipo_servico.value == 'baby':
            n_crianca = container_especifico.children[1].value
            gen = "Menino" if container_especifico.children[2].value == 'm' else "Menina"
            idade = container_especifico.children[3].value
            gostos_c = container_especifico.children[4].value
            saude = container_especifico.children[5].value

            if idade > 7:
                display(HTML("<div class='error-msg'>❌ Regra: Só aceitamos crianças até aos 7 anos.</div>"))
                return
            if not n_crianca or not saude:
                display(HTML("<div class='error-msg'>⚠️ Nome da criança e dados de Saúde/Dieta são obrigatórios.</div>"))
                return

            detalhes = f"<b>Criança:</b> {n_crianca} ({gen}) - {idade} anos<br><b>Gosta de:</b> {gostos_c}<br><b>Saúde/Dieta:</b> {saude}"

        # 3. Lógica Petsitting
        else:
            esp = container_especifico.children[1].value
            rac = container_especifico.children[2].value
            ida = container_especifico.children[3].value
            det = container_especifico.children[4].value

            if not esp or not det:
                display(HTML("<div class='error-msg'>⚠️ Indique a espécie e os detalhes de cuidados/dieta do animal.</div>"))
                return
            detalhes = f"<b>Pet:</b> {esp} ({rac}) - {ida} anos<br><b>Notas:</b> {det}"

        # EXIBIÇÃO DE SUCESSO
        display(HTML(f"""
            <div class='success-card'>
                <h3>✅ Pedido Registado com Confiança!</h3>
                <p><b>Responsável:</b> {nome_resp.value} | 📞 {contacto.value}</p>
                <p><b>Horário:</b> 15:00/16:00 às 17:00 ({dia_semana.label})</p>
                <hr style='opacity:0.2'>
                <p>{detalhes}</p>
                <p style='text-align:center; font-size:11px; margin-top:10px;'><i>A nossa equipa verificará a disponibilidade e ligará em breve.</i></p>
            </div>
        """))

btn.on_click(validar_e_exibir)

# --- RENDERIZAÇÃO FINAL ---
ui = widgets.VBox([
    widgets.HTML("<h1 class='title'>Kids & Pets Care</h1>"),
    tipo_servico,
    widgets.HTML("<div class='section-header'>Dados de Contacto</div>"),
    nome_resp, contacto, dia_semana,
    container_especifico,
    widgets.HTML("<br>"),
    btn, output
], layout=widgets.Layout(max_width='600px', margin='0 auto', padding='20px'))

display(ui)

In [8]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# --- ESTILO CSS PROFISSIONAL ---
display(HTML("""
<style>
    .app-container { border: 1px solid #d1d8e0; border-radius: 20px; padding: 40px; background-color: #ffffff; max-width: 600px; margin: 0 auto; box-shadow: 0 10px 25px rgba(0,0,0,0.1); font-family: 'Segoe UI', sans-serif; }
    .welcome-title { color: #2d98da; text-align: center; font-size: 28px; font-weight: bold; margin-bottom: 10px; }
    .role-btn { margin: 10px; width: 200px !important; height: 60px !important; font-size: 16px !important; font-weight: bold !important; }
    .auth-box { text-align: center; padding: 20px; }
</style>
"""))

output_principal = widgets.Output()

# --- FUNÇÃO PARA O FORMULÁRIO DO UTILIZADOR (O QUE JÁ TINHAMOS) ---
def mostrar_formulario_utilizador():
    with output_principal:
        clear_output()

        # Componentes do Formulário
        tipo = widgets.ToggleButtons(options=[('👶 Babysitting', 'baby'), ('🐾 Petsitting', 'pet')], description='Serviço:')
        nome_r = widgets.Text(description="Responsável:", placeholder="Seu nome")
        contacto = widgets.Text(description="Contacto:", placeholder="Emergência")

        # Container Dinâmico (Idade, Género, etc.)
        container_dyn = widgets.VBox()

        def atualizar_campos(change):
            if change['new'] == 'baby':
                container_dyn.children = [
                    widgets.Text(description="Nome Criança:", placeholder="Nome"),
                    widgets.Dropdown(options=['Menino', 'Menina'], description="Género:"),
                    widgets.IntText(description="Idade:", value=0),
                    widgets.Textarea(description="Interesses:", placeholder="O que gosta de fazer?"),
                    widgets.Textarea(description="Saúde/Dieta:", placeholder="ALERGIAS (Obrigatório)")
                ]
            else:
                container_dyn.children = [
                    widgets.Text(description="Espécie:", placeholder="Cão, Gato..."),
                    widgets.Text(description="Raça:", placeholder="Raça"),
                    widgets.IntText(description="Idade Pet:", value=0),
                    widgets.Textarea(description="Cuidados:", placeholder="Dieta e hábitos")
                ]

        tipo.observe(atualizar_campos, names='value')
        atualizar_campos({'new': 'baby'})

        btn_enviar = widgets.Button(description="SUBMETER RESERVA", button_style='success', layout=widgets.Layout(width='100%', height='45px'))
        btn_voltar = widgets.Button(description="Voltar ao Início", button_style='warning')

        def validar(b):
            # Lógica de validação (Idade < 7, etc.)
            # (Mantemos a lógica do código anterior aqui...)
            with output_principal:
                print("✅ Reserva enviada com sucesso!")

        btn_voltar.on_click(lambda x: mostrar_ecra_inicial())
        btn_enviar.on_click(validar)

        display(HTML("<h2 class='welcome-title'>Nova Reserva</h2>"))
        display(widgets.VBox([tipo, nome_r, contacto, container_dyn, widgets.HTML("<br>"), btn_enviar, btn_voltar]))

# --- FUNÇÃO PARA O ECRÃ DO TRABALHADOR ---
def mostrar_painel_trabalhador():
    with output_principal:
        clear_output()
        display(HTML("<h2 class='welcome-title'>Painel de Gestão</h2>"))
        display(HTML("<p style='text-align:center;'>Bem-vindo, colaborador. Aqui aparecerão as próximas marcações.</p>"))

        # Exemplo de lista de tarefas
        btn_voltar = widgets.Button(description="Sair", button_style='danger')
        btn_voltar.on_click(lambda x: mostrar_ecra_inicial())
        display(btn_voltar)

# --- FUNÇÃO DE AUTENTICAÇÃO ---
def verificar_password(b, pass_widget):
    if pass_widget.value == "babypetsiting2026":
        mostrar_painel_trabalhador()
    else:
        with output_principal:
            display(HTML("<p style='color:red; text-align:center;'>❌ Password Incorreta!</p>"))

def solicitar_password():
    with output_principal:
        clear_output()
        display(HTML("<div class='auth-box'><h3>Área Restrita</h3><p>Introduza a password de trabalhador:</p></div>"))
        pass_input = widgets.Password(placeholder='Password')
        btn_login = widgets.Button(description="Entrar", button_style='primary')
        btn_voltar = widgets.Button(description="Cancelar")

        btn_login.on_click(lambda b: verificar_password(b, pass_input))
        btn_voltar.on_click(lambda x: mostrar_ecra_inicial())

        display(widgets.VBox([pass_input, btn_login, btn_voltar], layout=widgets.Layout(align_items='center')))

# --- ECRÃ INICIAL (O QUE ABRE PRIMEIRO) ---
def mostrar_ecra_inicial():
    with output_principal:
        clear_output()
        display(HTML("""
            <div class='app-container'>
                <h1 class='welcome-title'>Kids & Pets Care</h1>
                <p style='text-align:center;'>Selecione o seu perfil para continuar</p>
                <hr>
            </div>
        """))

        btn_utilizador = widgets.Button(description="SOU UTILIZADOR", button_style='info', layout=widgets.Layout(width='250px', height='60px'))
        btn_trabalhador = widgets.Button(description="SOU TRABALHADOR", button_style='default', layout=widgets.Layout(width='250px', height='60px'))

        btn_utilizador.on_click(lambda x: mostrar_formulario_utilizador())
        btn_trabalhador.on_click(lambda x: solicitar_password())

        display(widgets.HBox([btn_utilizador, btn_trabalhador], layout=widgets.Layout(justify_content='center', padding='20px')))

# INICIAR APLICAÇÃO
mostrar_ecra_inicial()
display(output_principal)

Output()

In [9]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# --- ESTILO CSS ---
display(HTML("""
<style>
    .app-container { border: 2px solid #d1d8e0; border-radius: 20px; padding: 30px; background-color: #ffffff; max-width: 650px; margin: 0 auto; font-family: 'Segoe UI', sans-serif; }
    .title { color: #2d98da; text-align: center; font-size: 26px; margin-bottom: 20px; }
    .btn-main { font-weight: bold !important; height: 50px !important; }
    .error-box { background-color: #ffebee; color: #c62828; padding: 15px; border-radius: 8px; margin-top: 10px; border-left: 5px solid #c62828; }
    .success-box { background-color: #e8f5e9; color: #2e7d32; padding: 20px; border-radius: 8px; margin-top: 10px; border-left: 5px solid #4caf50; }
</style>
"""))

output_principal = widgets.Output()

# --- VARIÁVEIS DE CONTROLO ---
dados_reserva = {}

# --- ECRÃ 3: FORMULÁRIO DE UTILIZADOR ---
def mostrar_formulario_utilizador():
    with output_principal:
        clear_output()
        display(HTML("<h2 class='title'>📝 Nova Reserva</h2>"))

        # Campos fixos
        tipo = widgets.ToggleButtons(options=[('👶 Babysitting', 'baby'), ('🐾 Petsitting', 'pet')], description='Serviço:')
        nome_resp = widgets.Text(description="Responsável:", placeholder="Seu nome")
        contacto = widgets.Text(description="Emergência:", placeholder="Telemóvel")
        dia = widgets.Dropdown(options=[('Segunda a Sexta (15h-17h)', 'util'), ('Fim de Semana (Extra)', 'fds')], description="Dia:")

        container_dyn = widgets.VBox()

        # Função que gera os campos conforme a escolha
        def mudar_servico(change):
            if tipo.value == 'baby':
                container_dyn.children = [
                    widgets.HTML("<b>Dados do Menino/Menina:</b>"),
                    widgets.Text(description="Nome:", placeholder="Nome da criança"),
                    widgets.Dropdown(options=[('Menino', 'Menino'), ('Menina', 'Menina')], description="Género:"),
                    widgets.IntText(description="Idade:", value=0),
                    widgets.Textarea(description="Gosta de:", placeholder="O que gosta de fazer?"),
                    widgets.Textarea(description="Saúde/Dieta:", placeholder="Alergias e o que pode comer (OBRIGATÓRIO)")
                ]
            else:
                container_dyn.children = [
                    widgets.HTML("<b>Dados do Pet:</b>"),
                    widgets.Text(description="Espécie:", placeholder="Cão, Gato..."),
                    widgets.Text(description="Raça:", placeholder="Raça"),
                    widgets.IntText(description="Idade Pet:", value=0),
                    widgets.Textarea(description="Cuidados:", placeholder="Dieta e hábitos do animal")
                ]

        tipo.observe(mudar_servico, names='value')
        mudar_servico(None) # Inicializar campos

        btn_enviar = widgets.Button(description="FINALIZAR", button_style='success', layout=widgets.Layout(width='100%'))
        btn_voltar = widgets.Button(description="Voltar ao Início", layout=widgets.Layout(width='100%'))

        def validar(b):
            with output_principal:
                # Validação de campos vazios
                if not nome_resp.value or not contacto.value:
                    display(HTML("<p class='error-box'>⚠️ Nome do responsável e contacto são obrigatórios!</p>"))
                    return

                if tipo.value == 'baby':
                    nome_c = container_dyn.children[1].value
                    idade_c = container_dyn.children[3].value
                    saude_c = container_dyn.children[5].value

                    if idade_c > 7:
                        display(HTML("<p class='error-box'>❌ Erro: Só aceitamos crianças até aos 7 anos.</p>"))
                        return
                    if not nome_c or not saude_c:
                        display(HTML("<p class='error-box'>⚠️ Preencha o nome da criança e as alergias/dieta.</p>"))
                        return

                    resumo = f"✅ Reserva para <b>{nome_c}</b> enviada!"
                else:
                    resumo = "✅ Reserva de Petsitting enviada!"

                clear_output()
                display(HTML(f"<div class='success-box'>{resumo}<br>Entraremos em contacto brevemente.</div>"))
                display(btn_voltar)

        btn_enviar.on_click(validar)
        btn_voltar.on_click(lambda x: mostrar_ecra_inicial())

        display(widgets.VBox([tipo, nome_resp, contacto, dia, container_dyn, widgets.HTML("<br>"), btn_enviar, btn_voltar]))

# --- ECRÃ 2: LOGIN TRABALHADOR ---
def solicitar_password():
    with output_principal:
        clear_output()
        display(HTML("<h2 class='title'>🔐 Acesso Restrito</h2>"))
        pw = widgets.Password(description="Password:", placeholder="Digite aqui")
        btn_login = widgets.Button(description="Entrar", button_style='primary')
        btn_voltar = widgets.Button(description="Voltar")

        def check(b):
            if pw.value == "babypetsiting2026":
                with output_principal:
                    clear_output()
                    display(HTML("<div class='success-box'>🔓 Acesso Concedido. Bem-vindo ao Painel de Gestão 2026.</div>"))
                    btn_sair = widgets.Button(description="Sair", button_style='danger')
                    btn_sair.on_click(lambda x: mostrar_ecra_inicial())
                    display(btn_sair)
            else:
                display(HTML("<p style='color:red; text-align:center;'>Password errada!</p>"))

        btn_login.on_click(check)
        btn_voltar.on_click(lambda x: mostrar_ecra_inicial())
        display(widgets.VBox([pw, btn_login, btn_voltar], layout=widgets.Layout(align_items='center')))

# --- ECRÃ 1: INICIAL ---
def mostrar_ecra_inicial():
    with output_principal:
        clear_output()
        display(HTML("""
            <div class='app-container'>
                <h1 class='title'>✨ Kids & Pets Care</h1>
                <p style='text-align:center;'>Bem-vindo! Escolha o seu perfil:</p>
            </div>
        """))
        btn_u = widgets.Button(description="SOU UTILIZADOR", button_style='info', layout=widgets.Layout(width='45%', height='50px'))
        btn_t = widgets.Button(description="SOU TRABALHADOR", layout=widgets.Layout(width='45%', height='50px'))

        btn_u.on_click(lambda x: mostrar_formulario_utilizador())
        btn_t.on_click(lambda x: solicitar_password())

        display(widgets.HBox([btn_u, btn_t], layout=widgets.Layout(justify_content='center', padding='20px')))

# INÍCIO
mostrar_ecra_inicial()
display(output_principal)

Output()

In [11]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from datetime import datetime

# --- BASE DE DADOS TEMPORÁRIA ---
historico_reservas = []

# --- ESTILO CSS PROFISSIONAL COM BOTÃO WHATSAPP ---
display(HTML("""
<style>
    .app-container { border: 1px solid #d1d8e0; border-radius: 20px; padding: 25px; background-color: #ffffff; max-width: 700px; margin: 0 auto; font-family: 'Segoe UI', sans-serif; box-shadow: 0 10px 30px rgba(0,0,0,0.1); }
    .title { color: #2d98da; text-align: center; font-size: 26px; font-weight: bold; margin-bottom: 20px; }
    .whatsapp-btn { background-color: #25d366; color: white !important; padding: 10px 20px; border-radius: 50px; text-decoration: none; font-weight: bold; display: inline-block; margin-top: 20px; box-shadow: 0 4px 10px rgba(37,211,102,0.3); }
    .reserva-card { border-left: 5px solid #2d98da; background: #f9f9f9; padding: 15px; margin-bottom: 10px; border-radius: 5px; font-size: 13px; }
    .error-box { color: #e74c3c; font-weight: bold; padding: 10px; background: #fdeaea; border-radius: 5px; margin: 10px 0; }
    .success-box { color: #27ae60; font-weight: bold; padding: 10px; background: #eafaf1; border-radius: 5px; margin: 10px 0; }
</style>
"""))

output_principal = widgets.Output()

# --- COMPONENTES WHATSAPP ---
html_whatsapp = """
<center>
    <a href='https://wa.me/351912345678' target='_blank' class='whatsapp-btn'>
        💬 Contactar WhatsApp da Empresa
    </a>
    <p style='font-size: 11px; color: #7f8c8d; margin-top: 10px;'>Dúvidas? Fale connosco agora!</p>
</center>
"""

# --- ECRÃ DO TRABALHADOR (PAINEL DE GESTÃO) ---
def mostrar_painel_trabalhador():
    with output_principal:
        clear_output()
        display(HTML("<h2 class='title'>📊 Painel de Gestão (Trabalhador)</h2>"))

        if not historico_reservas:
            display(HTML("<p style='text-align:center;'>Nenhuma reserva registada até ao momento.</p>"))
        else:
            display(HTML(f"<p><b>Total de Reservas: {len(historico_reservas)}</b></p>"))
            for r in reversed(historico_reservas): # Mostrar as mais recentes primeiro
                tipo_icon = "👶" if r['tipo'] == 'baby' else "🐾"
                html_reserva = f"""
                <div class='reserva-card'>
                    <b>[{r['data']}] {tipo_icon} {r['servico']}</b><br>
                    <b>Responsável:</b> {r['responsavel']} ({r['contacto']})<br>
                    <b>Detalhes:</b> {r['detalhes']}<br>
                    <b>Saúde/Dieta:</b> {r['saude']}
                </div>
                """
                display(HTML(html_reserva))

        btn_sair = widgets.Button(description="Sair do Painel", button_style='danger')
        btn_sair.on_click(lambda x: mostrar_ecra_inicial())
        display(btn_sair)

# --- ECRÃ DO UTILIZADOR (FORMULÁRIO) ---
def mostrar_formulario_utilizador():
    with output_principal:
        clear_output()
        display(HTML("<h2 class='title'>📝 Pedido de Reserva</h2>"))

        tipo = widgets.ToggleButtons(options=[('👶 Babysitting', 'baby'), ('🐾 Petsitting', 'pet')], description='Serviço:')
        nome_resp = widgets.Text(description="Responsável:", placeholder="Seu nome")
        contacto_u = widgets.Text(description="Contacto:", placeholder="Telemóvel")
        dia_u = widgets.Dropdown(options=[('Segunda a Sexta (15h-17h)', 'util'), ('Fim de Semana (Extra)', 'fds')], description="Dia:")

        container_dyn = widgets.VBox()

        def atualizar_campos(change):
            if tipo.value == 'baby':
                container_dyn.children = [
                    widgets.Text(description="Nome Criança:", placeholder="Nome"),
                    widgets.Dropdown(options=['Menino', 'Menina'], description="Género:"),
                    widgets.IntText(description="Idade:", value=0),
                    widgets.Textarea(description="Gosta de:", placeholder="Atividades favoritas"),
                    widgets.Textarea(description="Saúde/Dieta:", placeholder="ALERGIAS e o que pode comer")
                ]
            else:
                container_dyn.children = [
                    widgets.Text(description="Espécie:", placeholder="Cão, Gato..."),
                    widgets.Text(description="Raça:", placeholder="Raça"),
                    widgets.IntText(description="Idade Pet:", value=0),
                    widgets.Textarea(description="Cuidados:", placeholder="Dieta e hábitos")
                ]

        tipo.observe(atualizar_campos, names='value')
        atualizar_campos({'new': 'baby'})

        btn_enviar = widgets.Button(description="ENVIAR RESERVA", button_style='success', layout=widgets.Layout(width='100%', height='45px'))
        btn_voltar = widgets.Button(description="Voltar ao Início", layout=widgets.Layout(width='100%'))

        def gravar_reserva(b):
            # Validação
            if not nome_resp.value or not contacto_u.value:
                with output_principal: display(HTML("<p class='error-box'>⚠️ Nome e Contacto são obrigatórios!</p>"))
                return

            # Criar dicionário com os dados
            nova_reserva = {
                'data': datetime.now().strftime("%d/%m/%Y %H:%M"),
                'tipo': tipo.value,
                'servico': "Babysitting" if tipo.value == 'baby' else "Petsitting",
                'responsavel': nome_resp.value,
                'contacto': contacto_u.value
            }

            if tipo.value == 'baby':
                if container_dyn.children[2].value > 7:
                    with output_principal: display(HTML("<p class='error-box'>❌ Apenas crianças até aos 7 anos.</p>"))
                    return
                nova_reserva['detalhes'] = f"{container_dyn.children[0].value} ({container_dyn.children[1].value}), {container_dyn.children[2].value} anos. Gosta de: {container_dyn.children[3].value}"
                nova_reserva['saude'] = container_dyn.children[4].value
            else:
                nova_reserva['detalhes'] = f"{container_dyn.children[0].value} ({container_dyn.children[1].value}), {container_dyn.children[2].value} anos."
                nova_reserva['saude'] = container_dyn.children[3].value

            # ADICIONAR À "BASE DE DADOS" GLOBAL
            historico_reservas.append(nova_reserva)

            with output_principal:
                clear_output()
                display(HTML("<div class='success-box'>✅ Reserva enviada! O trabalhador já recebeu os seus dados.</div>"))
                display(HTML(html_whatsapp))
                display(btn_voltar)

        btn_enviar.on_click(gravar_reserva)
        btn_voltar.on_click(lambda x: mostrar_ecra_inicial())

        display(widgets.VBox([tipo, nome_resp, contacto_u, dia_u, container_dyn, widgets.HTML("<br>"), btn_enviar, btn_voltar]))
        display(HTML(html_whatsapp)) # Botão Whatsapp sempre visível no formulário

# --- ECRÃ DE LOGIN (PASSWORD) ---
def solicitar_password():
    with output_principal:
        clear_output()
        display(HTML("<h2 class='title'>🔐 Área do Trabalhador</h2>"))
        pw = widgets.Password(description="Password:", placeholder="Insira a chave")
        btn_login = widgets.Button(description="Entrar", button_style='primary')
        btn_voltar = widgets.Button(description="Voltar")

        def check(b):
            if pw.value == "babypetsiting2026":
                mostrar_painel_trabalhador()
            else:
                display(HTML("<p style='color:red; text-align:center;'>Acesso Negado!</p>"))

        btn_login.on_click(check)
        btn_voltar.on_click(lambda x: mostrar_ecra_inicial())
        display(widgets.VBox([pw, btn_login, btn_voltar], layout=widgets.Layout(align_items='center')))

# --- ECRÃ INICIAL ---
def mostrar_ecra_inicial():
    with output_principal:
        clear_output()
        display(HTML("""
            <div class='app-container'>
                <h1 class='title'>✨ Kids & Pets Care</h1>
                <p style='text-align:center;'>Gestão de Babysitting e Petsitting 2026</p>
                <hr>
            </div>
        """))
        btn_u = widgets.Button(description="SOU UTILIZADOR", button_style='info', layout=widgets.Layout(width='280px', height='60px'))
        btn_t = widgets.Button(description="SOU TRABALHADOR", layout=widgets.Layout(width='280px', height='60px'))

        btn_u.on_click(lambda x: mostrar_formulario_utilizador())
        btn_t.on_click(lambda x: solicitar_password())

        display(widgets.HBox([btn_u, btn_t], layout=widgets.Layout(justify_content='center', padding='20px')))
        display(HTML(html_whatsapp))

# EXECUÇÃO INICIAL
mostrar_ecra_inicial()
display(output_principal)

Output()